In [ ]:
!pip install -q kagglehub
import kagglehub
kagglehub.login()

In [ ]:
path = kagglehub.dataset_download("dhivyeshrk/diseases-and-symptoms-dataset")
print("Dataset downloaded to:", path)

import os
os.listdir(path)

In [ ]:
import pandas as pd

csv_files = [f for f in os.listdir(path) if f.endswith(".csv")]
print("Files found:", csv_files)

df = pd.read_csv(os.path.join(path, csv_files[0]))
print(df.shape)
print(df.columns.tolist()[:10])  # confirm which column is the disease label
df.head()

In [ ]:
def normalize_symptom(value):
    import re
    if pd.isna(value):
        return None
    s = str(value).strip().lower()
    s = re.sub(r"\s+", "_", s)
    return s

label_col = df.columns[0]  # first column is the disease label in this dataset
SYMPTOM_COLS = [c for c in df.columns if c != label_col]

print("Label column:", label_col)
print(f"{len(SYMPTOM_COLS)} symptom columns")

df[label_col] = df[label_col].astype(str).str.strip()

symptom_vocab = [normalize_symptom(c) for c in SYMPTOM_COLS]
df = df.rename(columns=dict(zip(SYMPTOM_COLS, symptom_vocab)))

In [ ]:
from sklearn.ensemble import RandomForestClassifier

clf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1,  # use all available Colab CPU cores
)
clf.fit(X_train, y_train)
print("Trained.")

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = clf.predict(X_test)
v2_accuracy = accuracy_score(y_test, y_pred)
print("v2 (Random Forest) accuracy:", v2_accuracy)
print("v1 (Decision Tree) accuracy: 0.8162  <- hardcode your actual v1 number here")
print(f"Difference: {v2_accuracy - 0.8162:+.4f}")
print()

present_labels = sorted(set(y_test) | set(y_pred))
present_names = [label_encoder.classes_[i] for i in present_labels]
print(classification_report(y_test, y_pred, labels=present_labels, target_names=present_names, zero_division=0))

In [ ]:
importances = pd.Series(clf.feature_importances_, index=symptom_vocab).sort_values(ascending=False)
print("Top 20 most important symptoms:")
importances.head(20)

In [ ]:
candidates = {
    "n_estimators=500": RandomForestClassifier(n_estimators=500, random_state=42, n_jobs=-1),
    "max_depth=25": RandomForestClassifier(n_estimators=200, max_depth=25, random_state=42, n_jobs=-1),
}

results = {}
for name, model in candidates.items():
    model.fit(X_train, y_train)
    acc = accuracy_score(y_test, model.predict(X_test))
    results[name] = acc
    print(f"{name}: {acc:.4f}")

best_name = max(results, key=results.get)
print(f"\nBest variant: {best_name} ({results[best_name]:.4f})")
if results[best_name] > v2_accuracy:
    clf = candidates[best_name]  # keep the best-performing model for export
    print("Updated `clf` to the best-performing variant for export below.")

In [ ]:
import json
import joblib
import os

os.makedirs("model_artifacts", exist_ok=True)

joblib.dump(clf, "model_artifacts/v2_random_forest.joblib")

label_classes = [label_encoder.classes_[i] for i in clf.classes_]
with open("model_artifacts/label_classes.json", "w") as f:
    json.dump(label_classes, f, indent=2)

with open("model_artifacts/symptom_vocab.json", "w") as f:
    json.dump(symptom_vocab, f, indent=2)

assert len(label_classes) == len(clf.classes_)
assert clf.predict_proba(X_test[:1]).shape[1] == len(label_classes), "predict_proba width mismatch — do not ship"

print(f"{len(label_classes)} disease classes exported")
!ls -la model_artifacts

In [ ]:
import shutil
shutil.make_archive("v2_model_artifacts", "zip", "model_artifacts")

from google.colab import files
files.download("v2_model_artifacts.zip")